# Week 4 lab · Finding unusual card transactions with Isolation Forest

**Module:** Digital Innovation and Product Development · **Week 4:** Unsupervised learning · Work in teams.

## The situation

A bank has asked your team for a method that flags unusual card transactions, so that its analysts can look
at them. The method must use **Isolation Forest**.

You have been given 200,000 recent transactions. Nothing in the data says which transactions are anomalous.
So once you have built a model, how will you know whether it works?

To check a model you need at least some labelled examples. There is no single right way to get them: teams
use reports of past incidents, complaints from customers, or experts reviewing a random sample of
transactions. In this lab your team tries one practical option, which makes use of what you saw in the
lecture: a quick exploration with DBSCAN to decide where to look first. The plan:

1. **Keep 10% of the data aside as a test set.** Nobody looks at it until the very end.
2. **You need some labels** for the other 90% (the *development set*) to check and tune Isolation Forest.
   Reading 180,000 transactions one by one is not an option.
3. **Use DBSCAN to narrow the search.** Most transactions sit in one large, dense group. Transactions that
   DBSCAN marks as noise, or puts in very small clusters, do not sit with that group, so they are a sensible
   place to start looking.
4. **Inspect those candidates yourselves** and decide which look anomalous. Record your decisions by
   transaction ID. That gives you labels.
5. **Develop and tune Isolation Forest**, using your labels to measure how well it does.
6. **Run the finished model on the 10% you kept aside** and check what it flags.

```
 all transactions ──► 10% test set ────────────────────────────────────────────────► Task 4: final test
        │
        └──────────► 90% development set ──► Task 1: DBSCAN finds candidates
                                              ──► Task 2: you inspect them and label anomalies by ID
                                              ──► Task 3: develop and tune Isolation Forest on your labels
```

### Two things to keep in mind

* **DBSCAN does not tell you what is anomalous.** Noise points and small clusters are *candidates*: places
  where the data is sparse. Some will be harmless (a cardholder on holiday, say), and some anomalies may sit
  inside the large cluster, where DBSCAN will never point you. Your inspection, not DBSCAN, decides the
  labels.
* **Your labels are your team's judgement**, on the transactions you looked at. They are the best
  information you have, but they are not a complete or guaranteed answer.
* **This plan suits this data, not every problem.** It works best when anomalies stand apart from normal
  behaviour.

### Plan for the session (3.5 hours)

| Part | Time |
|---|---|
| Set-up: load, split and prepare the data | 20 min |
| Task 1 · Find candidates with DBSCAN | 45 min |
| Task 2 · Inspect the candidates and label anomalies | 40 min |
| Break | 10 min |
| Task 3 · Develop and tune Isolation Forest | 55 min |
| Task 4 · Final test on the held-out 10% | 40 min |

Cells marked **▶ RUN** work as they are. Cells marked **✎ YOUR CODE** or **✎ YOUR ANSWER** are for you.
Run the cells in order, from top to bottom.

## Set-up · load, split and prepare the data

In [ ]:
# ▶ RUN · libraries
# If a library is missing, install it in a notebook cell with:
#     %pip install scikit-learn pandas matplotlib
# (or from a terminal: pip install scikit-learn pandas matplotlib), then restart the kernel.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN               # Task 1
from sklearn.ensemble import IsolationForest     # Task 3
from sklearn.metrics import average_precision_score, precision_score, recall_score

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
SPLIT_SEED = 248      # fixes the 90/10 split, so every team gets the same test set

In [ ]:
# ▶ RUN · load the transactions (keep transactions.csv.gz in the same folder as this notebook)
tx = pd.read_csv("transactions.csv.gz")
print(f"{len(tx):,} transactions")
tx.head()

| Column | Meaning | Used by the models? |
|---|---|---|
| `txn_id` | transaction ID, to look up and label individual transactions | no |
| `amount_gbp` | amount in £ | **yes** |
| `hour` | time of day as a decimal hour (13.5 means 13:30) | **yes** |
| `distance_km` | distance between where the card was used and the cardholder's home area | **yes** |
| `mins_since_prev` | minutes since the same card's previous transaction | **yes** |
| `channel` | `in_store`, `online` or `atm` | no: for your inspection |
| `merchant_category` | kind of merchant | no: for your inspection |

DBSCAN and Isolation Forest work with **numbers only**, so the models use the four numeric columns. The text
columns `channel` and `merchant_category` are context: you will read them when you inspect transactions in
Task 2.

In [ ]:
# ▶ RUN · split once, at the start. The test set stays untouched until Task 4.
dev, test = train_test_split(tx, test_size=0.10, random_state=SPLIT_SEED)
dev = dev.reset_index(drop=True)
test = test.reset_index(drop=True)
print(f"development set: {len(dev):,} rows   test set: {len(test):,} rows")

### Preparing the numbers (done for you)

The four numeric columns are on very different scales: `distance_km` runs into thousands, `hour` only from
0 to 24. Amounts, distances and time gaps also have a few very large values. Left as they are, the biggest
numbers would dominate any distance between two transactions. The cell below fixes this for you, in two
standard steps:

1. take the logarithm of the three columns with very large values, so that £10 vs £100 counts as much as
   £100 vs £1,000;
2. rescale all four columns so that each has mean 0 and standard deviation 1.

The result is `X_dev` and `X_test`: one row per transaction (row *i* of `X_dev` is row *i* of `dev`), four
columns. **Both DBSCAN and Isolation Forest use these.** The rescaling is learnt from the development set
only, so nothing about the test set leaks into your work.

In [ ]:
# ▶ RUN · turn the numeric columns into model input
FEATURES = ["amount_gbp", "hour", "distance_km", "mins_since_prev"]
LOGGED = ["amount_gbp", "distance_km", "mins_since_prev"]

def log_features(df):
    out = df[FEATURES].copy()
    for col in LOGGED:
        out[col] = np.log1p(out[col])          # log(1 + x)
    return out

scaler = StandardScaler().fit(log_features(dev))      # learnt from dev only
X_dev = scaler.transform(log_features(dev))
X_test = scaler.transform(log_features(test))

fig, axes = plt.subplots(2, 4, figsize=(16, 5))
for j, col in enumerate(FEATURES):
    axes[0, j].hist(dev[col], bins=60, color="#b9c4c9"); axes[0, j].set_title(f"{col} (as recorded)")
    axes[1, j].hist(X_dev[:, j], bins=60, color="#2e5e6e"); axes[1, j].set_title(f"{col} (model input)")
    for ax in axes[:, j]: ax.set_yscale("log")
axes[0, 0].set_ylabel("transactions (log scale)"); axes[1, 0].set_ylabel("transactions (log scale)")
plt.tight_layout(); plt.show()
print("X_dev:", X_dev.shape, "  X_test:", X_test.shape)

## Task 1 · Find candidates with DBSCAN (45 min)

Goal: a short list of development-set transactions, about 150 or fewer, that do not sit with the bulk of
the data. These are the ones you will inspect in Task 2.

### 1.1 Choose Eps

DBSCAN needs two settings (see the lecture):
* **MinPts**, the number of neighbours a point needs to be in a dense area. We set it for you:
  **MinPts = 8** (a common rule of thumb is 2 × the number of features, and we have 4).
* **Eps**, how close those neighbours must be. You choose this one, from a chart.

The chart below shows, for each transaction, the distance to its 8th nearest neighbour, sorted from largest
to smallest, for the 1,000 transactions with the largest distances. On the left are the few transactions that
are far from everything else: the curve is high and drops steeply. Then it **flattens out**: from there on,
transactions have close neighbours. Choose Eps at the height where the curve flattens out (the "valley",
also called the knee or elbow).

In [ ]:
# ▶ RUN · the sorted k-dist chart (takes a few seconds)
MINPTS = 8
kdist = NearestNeighbors(n_neighbors=MINPTS).fit(X_dev).kneighbors(X_dev)[0][:, -1]
kdist = np.sort(kdist)[::-1][:1000]          # largest first; keep the first 1,000

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(np.arange(1, 1001), kdist, color="#2e5e6e", lw=2)
ax.set_xlabel("transactions, sorted by distance (largest first)")
ax.set_ylabel("distance to 8th nearest neighbour")
ax.set_yticks(np.arange(0, kdist.max() + 0.25, 0.25)); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ✎ YOUR CODE · 1.1 read Eps from the chart: the height where the curve flattens out
EPS = None

### 1.2 Run DBSCAN and look at the clusters

Run DBSCAN with one line: `db_labels = DBSCAN(eps=EPS, min_samples=MINPTS).fit_predict(X_dev)`.
It gives one label per transaction in `dev`: `-1` means noise, and `0, 1, 2, ...` are clusters.

On 180,000 rows this takes around 10–30 seconds. If it runs for minutes or runs out of memory, Eps is far
too large: look at the chart again.

In [ ]:
# ✎ YOUR CODE · 1.2 run DBSCAN on X_dev
db_labels = None

In [ ]:
# ▶ RUN · how many clusters, how big, and how many noise points?
assert db_labels is not None, "run DBSCAN in the cell above first"
sizes = pd.Series(db_labels[db_labels >= 0]).value_counts()
print(f"{len(sizes)} clusters; noise points: {(db_labels == -1).sum()}")
print("cluster sizes (cluster label: number of transactions):", sizes.to_dict())

### 1.3 Build the list of candidates

Decide what counts as a *small* cluster by setting `SMALL`. The rest of the cell is written for you: it
collects the candidates (all noise points plus all transactions in small clusters) into a table called
`candidates`, with an extra column `cluster` holding the DBSCAN label.

*Hint:* look at the cluster sizes from 1.2. You should see a few clusters with only a handful of
transactions, and one cluster holding almost all of them. There is a big gap between the two. Any value of
`SMALL` in that gap separates them, so you do not need an exact number.

**✎ YOUR ANSWER.** What threshold did you use for "small", and why?

*Your answer:*

In [ ]:
# ✎ YOUR CODE · 1.3 set SMALL, then run the cell to build the candidates
SMALL = None        # clusters with fewer transactions than this count as small

small_clusters = sizes[sizes < SMALL].index                       # labels of the small clusters
mask = (db_labels == -1) | np.isin(db_labels, small_clusters)     # noise, or in a small cluster
candidates = dev[mask].assign(cluster=db_labels[mask])            # those transactions, with their cluster label

In [ ]:
# ▶ RUN · check, and see where the candidates sit
assert isinstance(candidates, pd.DataFrame) and "cluster" in candidates, "build `candidates` with a `cluster` column first"
print(f"{len(candidates)} candidates out of {len(dev):,} development transactions ({len(candidates) / len(dev):.2%})")

sample = dev.sample(5000, random_state=0)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (x, y) in zip(axes, [("hour", "amount_gbp"), ("distance_km", "mins_since_prev")]):
    ax.scatter(sample[x], sample[y], s=4, color="#b9c4c9", label="5,000 random dev transactions")
    ax.scatter(candidates[x], candidates[y], s=14, color="#b5532b", label="candidates")
    ax.set_xlabel(x); ax.set_ylabel(y); ax.set_yscale("log")
    if x == "distance_km": ax.set_xscale("log")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, markerscale=3, frameon=False)
plt.tight_layout(rect=(0, 0.07, 1, 1)); plt.show()

**✎ YOUR ANSWER.** Look at the charts. Where do the candidates sit compared with the other transactions?

*Your answer:*

## Task 2 · Inspect the candidates and label anomalies (40 min)

Now look at the candidates and decide, as a team, which look anomalous. Use the context columns
(`channel`, `merchant_category`) as well as the numbers.

*How to work through them*
* Points in the same small cluster are similar: look at each small cluster as a group first.
* Sort the noise points (by amount, hour or distance) so that similar transactions sit next to each other.
* Use `show(ids)` below to display any set of candidates.

*A short guide, like the one the bank's analysts use*

| Usually worth flagging | Usually fine |
|---|---|
| Several tiny online payments within a minute or two of each other, often far from home: someone checking whether a stolen card works | A large purchase in a local shop during the day: a planned purchase |
| A large online payment or cash withdrawal, especially at night or far from home | Ordinary in-person payments abroad at ordinary times: the cardholder is travelling |

Not every candidate fits neatly into one of these. When you are unsure, discuss it, decide, and write down
why. A decision you can justify matters more than being certain.

In [ ]:
# ▶ RUN · helper to display candidates by ID, and a first overview
def show(ids, sort_by=None):
    rows = candidates[candidates.txn_id.isin(set(ids))]
    return rows.sort_values(sort_by) if sort_by else rows

display(candidates.groupby("cluster").agg(n=("txn_id", "size"), amount_min=("amount_gbp", "min"),
        amount_max=("amount_gbp", "max"), mins_since_prev_max=("mins_since_prev", "max"),
        channels=("channel", lambda c: ", ".join(sorted(set(c))))))
show(candidates.txn_id[candidates.cluster == -1], sort_by="amount_gbp").head(10)

In [ ]:
# ✎ YOUR CODE · 2.1 explore the candidates (add as many cells as you like)

### 2.2 Record your decisions

Fill in two sets of transaction IDs:
* `anomalous_ids`: the transactions you judged anomalous.
* `inspected_ids`: every transaction you looked at, whatever you decided (normally all the candidates).

You can type IDs in by hand (`{"T012345", "T067890"}`), or take them from a filtered table, for example
`set(candidates.txn_id[candidates.cluster == 3])`, as long as you looked at those rows and agree with them.

In [ ]:
# ✎ YOUR CODE · 2.2 record your decisions
anomalous_ids = set()                        # transaction IDs you judged anomalous
inspected_ids = set(candidates.txn_id)       # every transaction you inspected

### 2.3 Turn your decisions into labels

The function below adds two columns to a copy of `dev`:

| Column | Value |
|---|---|
| `inspected` | `True` if you looked at the transaction, `False` otherwise |
| `inspected_label` | `1` if you judged it anomalous, `0` otherwise |

**About the zeros.** `inspected_label = 0` means two different things: *inspected and judged normal*
(a few rows), or *never inspected* (almost all rows). In Task 3 we treat every 0 as normal, because we need
a complete column to measure against. That is an assumption, not a fact: some uninspected transactions may
well be anomalous. Keep this in mind when you read your results.

In [ ]:
# ▶ RUN · build the labelled development set from your two ID sets
def make_labelled_dataset(df, anomalous_ids, inspected_ids):
    unknown = set(anomalous_ids) - set(df.txn_id)
    assert not unknown, f"IDs not in this data set: {sorted(unknown)[:5]}"
    assert set(anomalous_ids) <= set(inspected_ids), "every anomalous ID should also be in inspected_ids"
    out = df.copy()
    out["inspected"] = out.txn_id.isin(inspected_ids)
    out["inspected_label"] = out.txn_id.isin(anomalous_ids).astype(int)
    return out

dev_labelled = make_labelled_dataset(dev, anomalous_ids, inspected_ids)
y_dev = dev_labelled.inspected_label.to_numpy()      # your labels as 0/1, in the same order as X_dev
print(f"inspected: {dev_labelled.inspected.sum()}   labelled anomalous: {y_dev.sum()}   "
      f"never inspected: {(~dev_labelled.inspected).sum():,}")
dev_labelled[dev_labelled.inspected].head()

## Task 3 · Develop and tune Isolation Forest (55 min)

Isolation Forest is **unsupervised**: it is fitted on `X_dev` only, never on the labels. Your labels `y_dev`
have one job here, which is to measure how well each version of the model does, so that you can choose
between them.

### 3.1 Fit Isolation Forest and score the development set

*Hints*
* `IsolationForest(random_state=0).fit(X_dev)` fits a model with the default settings.
* `model.score_samples(X_dev)` returns *lower* values for more anomalous points. Use
  `scores = -model.score_samples(X_dev)`, so that **higher = more anomalous**.

In [ ]:
# ✎ YOUR CODE · 3.1 fit a first model with the default settings and score dev
scores = None

### 3.2 Measure against your labels

**Flagging with a threshold.** An analyst sees only the transactions whose score is above a threshold. For
example, to flag the top 0.1%: `threshold = np.quantile(scores, 0.999)`, then `flagged = scores >= threshold`.
Every transaction is now either *flagged* or *not flagged*, and either *labelled anomalous* (1) or
*labelled normal* (0).

**The confusion matrix** counts the four combinations:

| | labelled anomalous (1) | labelled normal (0) |
|---|---|---|
| **flagged** | **TP**, true positive: an anomaly the model caught | **FP**, false positive: a false alarm |
| **not flagged** | **FN**, false negative: an anomaly the model missed | **TN**, true negative: correctly left alone |

Two numbers summarise it:

* **Precision = TP / (TP + FP)**: of the transactions the model flagged, the share that are anomalous.
  Low precision means analysts spend their time on false alarms.
* **Recall = TP / (TP + FN)**: of the anomalies, the share the model flagged. Low recall means anomalies
  slip through.

*Example.* There are 20 labelled anomalies. The model flags 10 transactions, and 8 of them are labelled
anomalous. Then TP = 8, FP = 2, FN = 20 − 8 = 12, so precision = 8 / 10 = 0.8 and recall = 8 / 20 = 0.4.

Raising the threshold flags fewer transactions: precision usually goes up and recall goes down. Lowering it
does the opposite. Choosing a threshold means choosing a balance between the two.

**Average precision (AP)** needs no threshold. It summarises precision over every possible threshold,
highest scores first: 1.0 means all your labelled anomalies score higher than everything else.
Use `average_precision_score(y_dev, scores)`.

Remember from 2.3 that "labelled normal" includes every transaction you never inspected.

In [ ]:
# ▶ RUN · helper: confusion matrix, precision and recall for a set of flags
def confusion(labels, flagged, what="labelled"):
    labels = np.asarray(labels).astype(bool)
    flagged = np.asarray(flagged).astype(bool)
    tp = int((flagged & labels).sum()); fp = int((flagged & ~labels).sum())
    fn = int((~flagged & labels).sum()); tn = int((~flagged & ~labels).sum())
    display(pd.DataFrame({f"{what} anomalous": [tp, fn], f"{what} normal": [fp, tn]}, index=["flagged", "not flagged"]))
    precision = tp / (tp + fp) if tp + fp else float("nan")
    recall = tp / (tp + fn) if tp + fn else float("nan")
    print(f"precision = TP / (TP + FP) = {tp} / {tp + fp} = {precision:.3f}")
    print(f"recall    = TP / (TP + FN) = {tp} / {tp + fn} = {recall:.3f}")
    return precision, recall

# Example: confusion(y_dev, scores >= threshold)

**✎ YOUR ANSWER.** In your own words: how many transactions did the model flag, and how many of them had
you labelled anomalous?

*Your answer:*

In [ ]:
# ✎ YOUR CODE · 3.2 compute AP, choose a threshold, and look at the confusion matrix with confusion(y_dev, flagged)

### 3.3 Tune

Try a few settings and compare. Keep the search small: a handful of values for each.
* `max_samples`: how many points each tree is built from (default 256).
* `n_estimators`: how many trees (default 100).
* `random_state`: run the same settings with 3 different seeds. If the results move a lot, the model is not
  stable yet.
* The threshold: how many transactions to flag. Plot precision and recall as you flag more and more.

Record what you tried in a small table (a list of dictionaries turned into a DataFrame works well).

In [ ]:
# ✎ YOUR CODE · 3.3 tuning: complete the two lines marked "...", then add other settings in the same way
results = []
for max_samples in [64, 256, 1024]:
    for seed in [0, 1, 2]:
        model = ...     # IsolationForest(max_samples=max_samples, random_state=seed).fit(X_dev)
        s = ...         # the scores, higher = more anomalous
        results.append({"max_samples": max_samples, "seed": seed, "AP": average_precision_score(y_dev, s)})
pd.DataFrame(results)

### 3.4 Choose your final model and threshold

**✎ YOUR ANSWER.** Which settings and which threshold did you choose? How many development transactions
does your threshold flag?

*Your answer:*

### 3.5 Hand over your final model

Complete `get_final_model()`. It must return the fitted model and your threshold. Task 4 calls it and
applies the model to `X_test`, so make sure it is final before you move on.

In [ ]:
# ✎ YOUR CODE · 3.5
def get_final_model():
    """Return (model, threshold): an IsolationForest fitted on X_dev, and the score above which to flag."""
    model = None        # fit on X_dev with your chosen settings
    threshold = None    # chosen on dev, from dev scores
    return model, threshold

## Task 4 · Final test on the held-out 10% (40 min)

Your model and threshold are fixed. Now, and only now, use the test set. The code below is provided: run it
without changing it. It scores the 20,000 test transactions with your model and lists the ones above your
threshold, highest score first. It also lists the ten transactions just *below* the threshold, the ones
your model nearly flagged.

In [ ]:
# ▶ RUN · score the test set with your final model
model, threshold = get_final_model()
assert model is not None and threshold is not None, "complete get_final_model() in Task 3 first"

test_scores = -model.score_samples(X_test)
test_flagged = test_scores >= threshold
print(f"test transactions: {len(test):,}   flagged by your model: {test_flagged.sum()} "
      f"({test_flagged.mean():.2%}; on dev you flagged {(-model.score_samples(X_dev) >= threshold).mean():.2%})")

ranked = test.assign(score=test_scores.round(3)).sort_values("score", ascending=False)
print("\nFlagged:")
display(ranked[ranked.score >= threshold])
print("Just below the threshold:")
display(ranked[ranked.score < threshold].head(10))

### 4.1 Check what was flagged

An analyst would now review the flagged transactions. Do the same: inspect each one with the guide from
Task 2 and record which you judge anomalous.

In [ ]:
# ✎ YOUR CODE · 4.1 IDs of flagged test transactions that you judge anomalous
test_anomalous_ids = set()

In [ ]:
# ▶ RUN · how much of what your model flagged did you judge anomalous?
n_flag = int(test_flagged.sum())
n_ok = len(set(test_anomalous_ids) & set(test.txn_id[test_flagged]))
print(f"flagged: {n_flag}   judged anomalous by your team (TP): {n_ok}   judged normal (FP): {n_flag - n_ok}")
print(f"precision on the test set, as judged by your team = TP / (TP + FP) = {n_ok} / {n_flag} = {n_ok / max(n_flag, 1):.2f}")

**✎ YOUR ANSWERS**

1. What precision did your review give on the test set? Is it close to what you saw on the development set?
2. Look at the transactions just below the threshold. Would you have wanted any of them flagged?

*Your answers:*